# 04 · Vegetation, Soil Moisture, and Heat — with Google Earth Engine

**What changed.** AppEEARS could take hours due to its processing queue. Google Earth Engine (GEE) delivers the same data (MODIS NDVI and SMAP soil moisture) in seconds because the intensive processing takes place on Google's servers, and only the results are downloaded to your computer. It is the **same NASA data sources**, accessed through a faster workflow.

**What's included in this notebook (broader coverage than before):**

- **NDVI and EVI** (MODIS MOD13Q1, 250 m, 2001–2025) — vegetation vigor.
- **Surface and root-zone soil moisture** (SMAP L4, 9–11 km, 2015–2025) — available soil water.
- **Land Surface Temperature (LST)** (MODIS MOD11A2, 1 km, 2001–2025) — thermal stress.

All variables are processed monthly for the 12 reference sites.

**Output:** `data/gee_monthly.csv`.

**Where this data goes (architecture):** The resulting CSV is uploaded to Supabase as precomputed data and used to power the platform. For new locations anywhere in the world, the application server will query GEE in real time using a service account (Phase 2).

In [2]:
import ee, sys
import pandas as pd, numpy as np
sys.path.append("../src")
import climate_tools as ct

ee.Initialize(project="kinetic-anvil-457906-e8")
print("Earth Engine:", ee.String("ok").getInfo())

SITES = {"Ipiales":(0.83,-77.64),"Nairobi":(-1.29,36.82),"Sorriso":(-12.54,-55.71),"Des Moines":(41.60,-93.61),
 "Ludhiana":(30.90,75.85),"Niamey":(13.51,2.11),"Can Tho":(10.03,105.78),"Mendoza":(-32.89,-68.84),"Fallon":(39.47,-118.78),
 "Ventaquemada":(5.37,-73.52),"Huancayo":(-12.07,-75.21),"Puno":(-15.84,-70.02)}
pts = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo,la]),{"site":n}) for n,(la,lo) in SITES.items()])

def monthly_ts(coll, band, scale, start, end, factor=1.0, offset=0.0):
    """Monthly mean of `band` at the 12 points, computed on Google's servers."""
    coll = coll.select(band).filterDate(start, end)
    n = ee.Date(end).difference(ee.Date(start), "month").toInt()
    months = ee.List.sequence(0, n.subtract(1))
    def mk(k):
        s = ee.Date(start).advance(k, "month")
        return coll.filterDate(s, s.advance(1, "month")).mean().multiply(factor).add(offset).set("month", s.format("YYYY-MM"))
    imgs = ee.ImageCollection(months.map(mk))
    def samp(img):
        return img.reduceRegions(pts, ee.Reducer.mean(), scale).map(lambda f: f.set("month", img.get("month")))
    rows = imgs.map(samp).flatten().getInfo()["features"]
    return pd.DataFrame([{"site":r["properties"].get("site"), "month":r["properties"].get("month"),
                          band:r["properties"].get("mean")} for r in rows])

Earth Engine: ok


In [3]:
# Pull everything (seconds). NDVI/EVI/LST from 2001; SMAP from 2015 (that is when SMAP begins).
print("descargando NDVI..."); ndvi = monthly_ts(ee.ImageCollection("MODIS/061/MOD13Q1"), "NDVI", 250, "2001-01-01", "2025-12-31", 0.0001)
print("descargando EVI...");  evi  = monthly_ts(ee.ImageCollection("MODIS/061/MOD13Q1"), "EVI", 250, "2001-01-01", "2025-12-31", 0.0001)
print("descargando temperatura de superficie..."); lst = monthly_ts(ee.ImageCollection("MODIS/061/MOD11A2"), "LST_Day_1km", 1000, "2001-01-01", "2025-12-31", 0.02, -273.15)
print("descargando humedad superficial..."); sm_s = monthly_ts(ee.ImageCollection("NASA/SMAP/SPL4SMGP/008"), "sm_surface", 11000, "2015-04-01", "2025-12-31")
print("descargando humedad de raíz...");      sm_r = monthly_ts(ee.ImageCollection("NASA/SMAP/SPL4SMGP/008"), "sm_rootzone", 11000, "2015-04-01", "2025-12-31")

gee = ndvi
for df in (evi, lst, sm_s, sm_r):
    gee = gee.merge(df, how="outer", on=["site","month"])
gee = gee.rename(columns={"NDVI":"ndvi","EVI":"evi","LST_Day_1km":"lst_c","sm_surface":"sm_surf","sm_rootzone":"sm_root"}).sort_values(["site","month"])
gee.to_csv("../data/gee_monthly.csv", index=False)
print("\nfilas:", len(gee), "| meses NDVI:", ndvi.month.nunique(), "| meses SMAP:", sm_s.month.nunique())
print("\nPromedios por sitio (NDVI, humedad raíz m3/m3, temp superficie C):")
print(gee.groupby("site")[["ndvi","sm_root","lst_c"]].mean().round(2).to_string())

descargando NDVI...
descargando EVI...
descargando temperatura de superficie...
descargando humedad superficial...
descargando humedad de raíz...

filas: 3600 | meses NDVI: 299 | meses SMAP: 129

Promedios por sitio (NDVI, humedad raíz m3/m3, temp superficie C):
              ndvi  sm_root  lst_c
site                              
Can Tho       0.19     0.24  33.55
Des Moines    0.35     0.34  19.42
Fallon        0.24     0.03  27.95
Huancayo      0.10     0.21  25.53
Ipiales       0.22     0.91  23.85
Ludhiana      0.26     0.20  30.89
Mendoza       0.18     0.09  23.37
Nairobi       0.28     0.33  28.90
Niamey        0.27     0.05  34.22
Puno          0.15     0.18  25.96
Sorriso       0.52     0.36  35.19
Ventaquemada  0.56     0.25  20.19
